# Contexte

L'objectif de ce notebook est de proceder au pré-traitement des données suite à notre projet DPM portant sur l'étude de review issue du jeu de données de Tristpilot.   
Nous essaierons au cours du notebook de proposer une pipeline propre au traitement de langage naturel.

In [325]:
import pandas as pd 

In [326]:
df = pd.read_csv('trustpilot_reviews_2005.csv')

In [327]:
df

,category,company,description,title,review,stars
0,Animals & Pets,ruffandtumbledogcoats.com,At Ruff and Tumble we are proud to be the mark...,Great quality dog drying robe although…,Great quality dog drying robe although had to ...,5
1,Animals & Pets,ruffandtumbledogcoats.com,At Ruff and Tumble we are proud to be the mark...,Really prompt service,"Really prompt service, The sofa covers have no...",5
2,Animals & Pets,ruffandtumbledogcoats.com,At Ruff and Tumble we are proud to be the mark...,Life saver,I’ve purchased first of those coats in May2020...,5
3,Animals & Pets,ruffandtumbledogcoats.com,At Ruff and Tumble we are proud to be the mark...,Brilliant coats,Brilliant coats. Really like the limited editi...,5
4,Animals & Pets,ruffandtumbledogcoats.com,At Ruff and Tumble we are proud to be the mark...,Great company and products,Great company and products. This is my 3rd dry...,5
...,...,...,...,...,...,...
123176,Media & Publishing,rarewaves.com,Rarewaves.com are one of the largest online re...,Still haven't received my purchase even…,Still haven't received my purchase even though...,1
123177,Media & Publishing,rarewaves.com,Rarewaves.com are one of the largest online re...,Ordered 10 days ago.....not a peep,Ordered 10 days ago thinking it would be here ...,1
123178,Media & Publishing,rarewaves.com,Rarewaves.com are one of the largest online re...,Bad first experience,Discs arrived broken and I still had to email ...,1
123179,Media & Publishing,rarewaves.com,Rarewaves.com are one of the largest online re...,Waiting .....2 mths now..never again,Waiting .....2 mths now..never again. No track...,1


In [384]:
# Afficher les catégories qui nous intéressent : ici Restaurant et Bars
df['category'].unique() 

array(['Animals & Pets', 'Sports', 'Beauty & Well-being',
       'Hobbies & Crafts', 'Health & Medical', 'Shopping & Fashion',
       'Home & Garden', 'Electronics & Technology',
       'Food, Beverages & Tobacco', 'Home Services', 'Business Services',
       'Restaurants & Bars', 'Money & Insurance', 'Travel & Vacation',
       'Construction & Manufacturing', 'Utilities',
       'Vehicles & Transportation', 'Media & Publishing',
       'Public & Local Services', 'Education & Training',
       'Legal Services & Government', 'Events & Entertainment'],
      dtype=object)

In [385]:
#Création de notre jeu de données principal
df_resto = df[df['category'] == 'Restaurants & Bars']

# Pre-processing
Fonction dummy qui à pour but de supprimer les espaces

In [331]:
def dummy_word_split(texts):

    texts_out = []
    for text in texts:
        texts_out.append(text.split(" "))

    return texts_out

In [332]:
splitted_texts = dummy_word_split(df_resto['review'].tolist())

In [333]:
splitted_texts[0]

['Big',
 'up',
 'to',
 'my',
 'guy',
 'Moses',
 'who',
 'works',
 'in',
 'the',
 'itsu',
 'in',
 'Gatwick',
 'airport',
 'kind,gentle',
 'and',
 'welcoming',
 'and',
 'made',
 'my',
 'journey',
 'even',
 'better',
 'when',
 'him',
 'putting',
 'on',
 'a',
 'smile',
 'Thanks',
 'Moses',
 'continue',
 'to',
 'shine',
 'light',
 'on',
 'others',
 '🤞']

In [334]:
import itertools
from itertools import chain

def compute_word_occurence(texts):

    words = itertools.chain.from_iterable(texts)
    word_count = pd.Series(words).value_counts()
    word_count = pd.DataFrame({"Word": word_count.index, "Count": word_count.values})

    return word_count

In [335]:
compute_word_occurence(splitted_texts).head(20)

,Word,Count
0,the,13869
1,and,11515
2,to,9014
3,a,7521
4,I,6921
5,was,6372
6,of,4966
7,for,4112
8,,3671
9,in,3239


# Obsevations 

Ici, nous observons que les mots les plus utilisés sont des stop-words : pronoms, déterminants, prépositions... Ces mots sont très nombreux et apportent peu d'information concernant les reviews : notre objectif premier sera des les supprimer.  
D'autres observations comme des répetitions dues aux majuscules : The/the, et des mots signifiant la même racine mais exprimer de manière différente : have/had, nous les traiterons plus bas.

In [386]:
#Qualité des données : les données contiennent-elles des string/Nan ?
import numpy as np

def quality(texts):

    assert all([isinstance(t,str) for t in texts]), "Présence de caractère non string"
    assert all([ t!= np.nan for t in texts]), "Présence de NaN"

    return True

In [337]:
def force_format(texts):
    return [str(t) for t in texts]

In [338]:
texts = force_format(df_resto['review'])

In [339]:
print(f" Si True pas de valeurs autres que String et pas de NaN, présence sinon\n {quality(texts)}")

 Si True pas de valeurs autres que String et pas de NaN, présence sinon
 True


Nos données ne comportent pas de caractère non string et pas de NaN

In [340]:
"""
import sys
!{sys.executable} -m pip install emoji

Importation de la librairie emoji afin de les supprimer dans les textes
"""

'\nimport sys\n!{sys.executable} -m pip install emoji\n\nImportation de la librairie emoji afin de les supprimer dans les textes\n'

In [341]:
#FOnction de suppression de balise/carractère spéciaux
import re
import emoji 

def filter_text(texts_in):
    
    texts_out = re.sub(r'https?:\/\/[A-Za-z0-9_.-~\-]*', ' ', texts_in, flags=re.MULTILINE)
    texts_out = re.sub(r'[(){}\[\]<>]', ' ', texts_out, flags=re.MULTILINE)
    texts_out = re.sub(r'&amp;#.*;', ' ', texts_out, flags=re.MULTILINE)
    texts_out = re.sub(r'&gt;', ' ', texts_out, flags=re.MULTILINE)
    texts_out = re.sub(r'â€™', "'", texts_out, flags=re.MULTILINE)
    texts_out = re.sub(r'\s+', ' ', texts_out, flags=re.MULTILINE)
    texts_out = re.sub(r'&#x200B;', ' ', texts_out, flags=re.MULTILINE)
    texts_out = re.sub(r"\(?\d{3}\D{0,3}\d{3}\D{0,3}\d{4}", '', texts_out, flags=re.MULTILINE)
    texts_out = emoji.replace_emoji(texts_out, replace='')
    texts_out = texts_out.lower()

    return texts_out


Nous avons observés plus haut la présence d'emojis qui n'apportent pas d'information dans le traitement de texte, bien que le jeu
de données semble correct aux niveaux des balises, cette méthode est généralement utilsée afin de supprimer des carractères non souhaités avec des Regex.  

Nous mettons nos carractères en minuscule afin de palier vu lors de la première phase.

In [342]:
texts = [filter_text(t) for t in texts]

In [343]:
texts

['big up to my guy moses who works in the itsu in gatwick airport kind,gentle and welcoming and made my journey even better when him putting on a smile thanks moses continue to shine light on others ',
 'great experience at itsu thanks to hospitality of moses at gatwick airport south terminal',
 'as itsu virgins we enjoyed a lovely first time experience at the itsu liverpool street. the staff were exceptionally good at helping us navigate our way through the ordering process, the food was hot, tasty and efficiently produced. we’ll certainly try itsu again!',
 'itsu at brunswick centre, camden is an extremely nice sushi place. the staff are very helpful especially ornella and wojciech who have taught me how to operate the technology. now i can scan with my app and claim my butterflies with ease. i have also been taught how to access my nhs discount. thank goodness for lovely staff who enjoy their jobs and make customers feel special.',
 'at itsu reading gate and clean restaurant and fri

## Unification du texte & conversion des phrase en liste de mot

Spécialement prévu pour notre projet de Topic Modeling : nous ne souhaitons pas savoir si un mot se situe au début ou à la fin d'une phrase. Nous avons déjà supprimé les majuscules, nous procédons plus bas à la suppression des accents (bien que peut présent dans la langue anglaise).


In [344]:
"""
import sys
!{sys.executable} -m pip install tqdm
"""

'\nimport sys\n!{sys.executable} -m pip install tqdm\n'

In [345]:
"""
import sys
!{sys.executable} -m pip install gensim
"""

'\nimport sys\n!{sys.executable} -m pip install gensim\n'

In [346]:
from tqdm import tqdm
from gensim.utils import simple_preprocess

def sent_to_words(sentences):
    for sentence in tqdm(sentences):
        yield simple_preprocess(str(sentence), deacc=True)

In [347]:
texts = list(sent_to_words(texts))

100%|██████████| 5204/5204 [00:00<00:00, 8351.21it/s]


In [348]:
texts

[['big',
  'up',
  'to',
  'my',
  'guy',
  'moses',
  'who',
  'works',
  'in',
  'the',
  'itsu',
  'in',
  'gatwick',
  'airport',
  'kind',
  'gentle',
  'and',
  'welcoming',
  'and',
  'made',
  'my',
  'journey',
  'even',
  'better',
  'when',
  'him',
  'putting',
  'on',
  'smile',
  'thanks',
  'moses',
  'continue',
  'to',
  'shine',
  'light',
  'on',
  'others'],
 ['great',
  'experience',
  'at',
  'itsu',
  'thanks',
  'to',
  'hospitality',
  'of',
  'moses',
  'at',
  'gatwick',
  'airport',
  'south',
  'terminal'],
 ['as',
  'itsu',
  'virgins',
  'we',
  'enjoyed',
  'lovely',
  'first',
  'time',
  'experience',
  'at',
  'the',
  'itsu',
  'liverpool',
  'street',
  'the',
  'staff',
  'were',
  'exceptionally',
  'good',
  'at',
  'helping',
  'us',
  'navigate',
  'our',
  'way',
  'through',
  'the',
  'ordering',
  'process',
  'the',
  'food',
  'was',
  'hot',
  'tasty',
  'and',
  'efficiently',
  'produced',
  'we',
  'll',
  'certainly',
  'try',
  'its

## Suppression des STOPWORDS

Comme évoqué plus haut, certains mots sont très redondants et n'apportent pas d'information : nous importons grâce à la bibliothèque Spacy, une liste prédéfinie des stopwords que nous supprimons par la suite.

In [313]:
"""
import sys
!{sys.executable} -m pip install spacy==3.7.5 
#les autres versions ne fonctionnent pas lors de l'import
"""

"\nimport sys\n!{sys.executable} -m pip install spacy==3.7.5 \n#les autres versions ne fonctionnent pas lors de l'import\n"

In [349]:
from spacy.lang.en.stop_words import STOP_WORDS as en_stop

In [350]:
#Définitions des stopwords prédéfinis à supprimer
texts = [
    [word for word in doc if word not in en_stop]
    for doc in texts
]
            

In [351]:
texts

[['big',
  'guy',
  'moses',
  'works',
  'itsu',
  'gatwick',
  'airport',
  'kind',
  'gentle',
  'welcoming',
  'journey',
  'better',
  'putting',
  'smile',
  'thanks',
  'moses',
  'continue',
  'shine',
  'light'],
 ['great',
  'experience',
  'itsu',
  'thanks',
  'hospitality',
  'moses',
  'gatwick',
  'airport',
  'south',
  'terminal'],
 ['itsu',
  'virgins',
  'enjoyed',
  'lovely',
  'time',
  'experience',
  'itsu',
  'liverpool',
  'street',
  'staff',
  'exceptionally',
  'good',
  'helping',
  'navigate',
  'way',
  'ordering',
  'process',
  'food',
  'hot',
  'tasty',
  'efficiently',
  'produced',
  'll',
  'certainly',
  'try',
  'itsu'],
 ['itsu',
  'brunswick',
  'centre',
  'camden',
  'extremely',
  'nice',
  'sushi',
  'place',
  'staff',
  'helpful',
  'especially',
  'ornella',
  'wojciech',
  'taught',
  'operate',
  'technology',
  'scan',
  'app',
  'claim',
  'butterflies',
  'ease',
  'taught',
  'access',
  'nhs',
  'discount',
  'thank',
  'goodness'

Ici, on voit qu'on perd le "up" de "big up", qui peut prêter à confusion, on fait le choix de tout de même continuer 
et de créer les n-grams a posteriori car nous conservons la valeurs positive de la phrase 

# Création de n-grams
Si nous prenons New York par exemple, ce mot est en fait composé de deux noms disctincts. Au final, notre fonction word_count ne prendra pas en compte le mot en une fois mais "New" et "York", afin de palier à ce problème la création de bigram/trigram nous permet de lier les mots qui se retrouvent souvent entre eux : on identifie les mots qui sont souvent collés et nous les regroupons : augmentation de l'interprétabilité pour le topic modeling.

In [317]:
from gensim.models.phrases import Phraser
from gensim.models import Word2Vec, Phrases, KeyedVectors



In [359]:
# Création de bigram 
def create_bigrams(texts, min_count=5, threshold=5, as_str=False):

    bigram_model = Phraser(
        Phrases(texts, min_count=min_count, threshold=threshold)
    )

    texts_bigram = [bigram_model[doc] for doc in texts]

    if as_str:
        return [" ".join(doc) for doc in texts_bigram]

    return texts_bigram

#Création de trigram
def create_trigrams(texts, min_count_bigram=15,
                    min_count_trigram=10,
                    threshold=10,
                    as_str=False):

    # Bigrams
    bigram_model = Phraser(
        Phrases(texts,
                min_count=min_count_bigram,
                threshold=threshold)
    )

    texts_bigram = [bigram_model[doc] for doc in texts]

    # Trigrams
    trigram_model = Phraser(
        Phrases(texts_bigram,
                min_count=min_count_trigram,
                threshold=threshold)
    )

    texts_trigram = [trigram_model[doc] for doc in texts_bigram]

    if as_str:
        return [" ".join(doc) for doc in texts_trigram]

    return texts_trigram

In [360]:
texts = create_bigrams(texts)
texts = create_trigrams(texts)

In [361]:
texts

[['big',
  'guy',
  'moses',
  'works',
  'itsu',
  'gatwick',
  'airport',
  'kind',
  'gentle',
  'welcoming',
  'journey',
  'better',
  'putting',
  'smile',
  'thanks',
  'moses',
  'continue',
  'shine',
  'light'],
 ['great_experience',
  'itsu',
  'thanks',
  'hospitality',
  'moses',
  'gatwick',
  'airport',
  'south',
  'terminal'],
 ['itsu',
  'virgins',
  'enjoyed',
  'lovely',
  'time',
  'experience',
  'itsu',
  'liverpool',
  'street',
  'staff',
  'exceptionally',
  'good',
  'helping',
  'navigate',
  'way',
  'ordering_process',
  'food',
  'hot',
  'tasty',
  'efficiently',
  'produced',
  'll',
  'certainly',
  'try',
  'itsu'],
 ['itsu',
  'brunswick',
  'centre',
  'camden',
  'extremely',
  'nice',
  'sushi',
  'place',
  'staff_helpful',
  'especially',
  'ornella',
  'wojciech',
  'taught',
  'operate',
  'technology',
  'scan',
  'app',
  'claim',
  'butterflies',
  'ease',
  'taught',
  'access',
  'nhs',
  'discount',
  'thank',
  'goodness',
  'lovely',
 

absolutely_love, highly_recommend, sont identifiés : Nous permettra de facilité la tâche lors de l'analyse de sentimenet / clustering des reviews.

In [321]:
"""
import sys
!{sys.executable} -m pip install spacy
!{sys.executable} -m spacy download en_core_web_sm
"""

'\nimport sys\n!{sys.executable} -m pip install spacy\n!{sys.executable} -m spacy download en_core_web_sm\n'

## Lemmatization & Stemming

run, running, runnable sont plusieurs mots qui ne devraient être compter que comme un seul lors de la création de Topic. 

Le Stemming est une méthode qui permet de couper les mots à leur racine de manière assez brutale (studies => studi) et peut apporter un manque de précision.  

La Lemmatization nous permet de s'approprier les règles grammaticales des mots afin de mieux les regrouper (running => run, better => good). Nous obtiendrons une fonction de comptage des mots plus précise avec cette méthode.  

La librairie Spacy nous aide encore une fois ici à appliquer cette méthode.


In [362]:
print(texts[0])

['big', 'guy', 'moses', 'works', 'itsu', 'gatwick', 'airport', 'kind', 'gentle', 'welcoming', 'journey', 'better', 'putting', 'smile', 'thanks', 'moses', 'continue', 'shine', 'light']


In [ ]:
def lemmatize_texts(texts, allowed_postags, spacy_model=None):

    if not spacy_model:
        spacy_model = spacy.load("en_core_web_sm")

    texts = [
        " ".join(doc) if isinstance(doc, list) else str(doc)
        for doc in texts
    ]

    docs = spacy_model.pipe(texts)

    return [
        [t.lemma_ for t in doc if t.pos_ in allowed_postags]
        for doc in docs
    ]

In [376]:
# Nous choissons de conserver les noms communs, noms propres, les verbes, adjectifs, adverbes ainsi que les termes non reconnus et spéciaux avec 'X'
l_texts = lemmatize_texts(texts_spacy,
                allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV', 'X', 'PROPN'])

In [382]:
count = compute_word_occurence(l_texts)


In [383]:
count.head(50)

,Word,Count
0,order,2314
1,food,2008
2,good,1372
3,time,1150
4,meal,1094
5,delivery,966
6,great,866
7,go,825
8,service,688
9,box,682


## Observations 

Nous observons maintenant que nos mots se concentrent bien sur des sujets cohérents avec notre thème "Restaurant & Bars" après la mise en place de notre pipeline de traitement de langage.  

Notre comptage de mot ne contient plus de stopwords, de double occurence et possède même des bigrams (customer_service) qui nous seront utiles pour nos prochaines analyses.

In [387]:
l_texts

[['big',
  'guy',
  'moses',
  'work',
  'itsu',
  'gatwick',
  'airport',
  'kind',
  'gentle',
  'welcoming',
  'journey',
  'well',
  'put',
  'smile',
  'thank',
  'moses',
  'continue',
  'shine',
  'light'],
 ['great_experience',
  'itsu',
  'thank',
  'hospitality',
  'moses',
  'gatwick',
  'airport',
  'south',
  'terminal'],
 ['itsu',
  'virgin',
  'enjoy',
  'lovely',
  'time',
  'experience',
  'itsu',
  'liverpool',
  'street',
  'staff',
  'exceptionally',
  'good',
  'helping',
  'navigate',
  'way',
  'ordering_process',
  'food',
  'hot',
  'tasty',
  'efficiently',
  'produce',
  'certainly',
  'try',
  'itsu'],
 ['itsu',
  'brunswick',
  'centre',
  'camden',
  'extremely',
  'nice',
  'sushi',
  'place',
  'staff_helpful',
  'especially',
  'ornella',
  'wojciech',
  'teach',
  'operate',
  'technology',
  'scan',
  'app',
  'claim',
  'butterfly',
  'ease',
  'teach',
  'access',
  'nhs',
  'discount',
  'thank',
  'goodness',
  'lovely',
  'staff',
  'enjoy',
  'j

On ajoute notre colonne de phrases lemmatizées à notre DataFrame.

In [389]:
df_resto["l_texts"] = l_texts

C:\Users\Rudyl\AppData\Local\Temp/ipykernel_19568/3679679886.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_resto["l_texts"] = l_texts


In [390]:
df_resto

,category,company,description,title,review,stars,l_texts
49908,Restaurants & Bars,itsu.com,"Delicious, healthy, Asian inspired food, fresh...",Big up to my guy Moses who works in the…,Big up to my guy Moses who works in the itsu i...,5,"[big, guy, moses, work, itsu, gatwick, airport..."
49909,Restaurants & Bars,itsu.com,"Delicious, healthy, Asian inspired food, fresh...",Great experience at itsu thanks to…,Great experience at itsu thanks to hospitality...,5,"[great_experience, itsu, thank, hospitality, m..."
49910,Restaurants & Bars,itsu.com,"Delicious, healthy, Asian inspired food, fresh...",As Itsu virgins we enjoyed a lovely…,As Itsu virgins we enjoyed a lovely first time...,5,"[itsu, virgin, enjoy, lovely, time, experience..."
49911,Restaurants & Bars,itsu.com,"Delicious, healthy, Asian inspired food, fresh...",ITSU at Brunswick Centre: three cheers for the...,"ITSU at Brunswick Centre, Camden is an extreme...",5,"[itsu, brunswick, centre, camden, extremely, n..."
49912,Restaurants & Bars,itsu.com,"Delicious, healthy, Asian inspired food, fresh...",Itsu Reading Gate is Great,At Itsu Reading Gate and clean restaurant and ...,5,"[itsu, read, gate, clean, restaurant, food, qu..."
...,...,...,...,...,...,...,...
112775,Restaurants & Bars,littleleeds.co.uk,A modern and chic play café for grown ups to e...,Not bad!,We loved the idea and our boy enjoyed it!!! No...,3,"[love, idea, boy, enjoy, keen, birthday_party,..."
112776,Restaurants & Bars,littleleeds.co.uk,A modern and chic play café for grown ups to e...,Disappointing,I really want to like this place but I echo ot...,3,"[want, place, echo, people, feeling, people, a..."
112777,Restaurants & Bars,littleleeds.co.uk,A modern and chic play café for grown ups to e...,Really lovely little place however as…,Really lovely little place however as you ente...,3,"[lovely, little, place, enter, array, sugary, ..."
112778,Restaurants & Bars,littleleeds.co.uk,A modern and chic play café for grown ups to e...,Disappointed,This was a disappointing trip as it was very s...,3,"[disappointing, trip, small, venue, hardly, sp..."


Afin d'analyser et d'affectuer des modélisations plus approfondies, la colonnes l_texts nous sera très utiles.

In [ ]:
pd.to_csv(